# IT3051 - Fundamentals of Data Mining & Machine Learning
## Mini Project 2026: Used Car Price Prediction System
### Stage 7: Comprehensive Model Comparison, Evaluation & Final Selection

This notebook:
1. Gathers actual experimental results from all four machine learning models:
   - **Linear Regression** (OLS Baseline)
   - **Decision Tree** (Baseline & Tuned)
   - **Random Forest** (Baseline, Tuned, Feature-Selected)
   - **Gradient Boosting** (Baseline & Tuned)
2. Compiles the master **Experiment Log** (`results/experiment_log.csv`).
3. Generates comprehensive academic visual diagnostics.
4. Conducts multi-metric comparative analysis.
5. Declares and justifies the final champion model configuration.

In [ ]:
# 1. Import Core Libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.float_format', '{:,.4f}'.format)

---
## 2. Compile Cross-Model Performance Table

In [ ]:
# Collect individual results
model_files = [
    os.path.join('results', 'linear_regression_results.csv'),
    os.path.join('results', 'decision_tree_results.csv'),
    os.path.join('results', 'random_forest_results.csv'),
    os.path.join('results', 'gradient_boosting_results.csv')
]

all_dfs = [pd.read_csv(f) for f in model_files if os.path.exists(f)]
combined_df = pd.concat(all_dfs, ignore_index=True)
combined_df.insert(0, 'Experiment_ID', [f"EXP_{i+1:02d}" for i in range(len(combined_df))])

# Save experiment log and summary tables
combined_df.to_csv(os.path.join('results', 'experiment_log.csv'), index=False)
combined_df[combined_df['Version'] == 'Baseline'].to_csv(os.path.join('results', 'baseline_results.csv'), index=False)
combined_df[combined_df['Version'] == 'Tuned'].to_csv(os.path.join('results', 'tuned_results.csv'), index=False)

display_cols = ['Experiment_ID', 'Model', 'Version', 'CV_R2', 'CV_MAE', 'CV_RMSE', 'Test_R2', 'Test_MAE', 'Test_RMSE', 'Test_MAE_Dollar']
print("=== Cross-Model Performance Comparison Table ===")
print(combined_df[display_cols].to_string(index=False))

---
## 3. Visual Performance Comparisons

In [ ]:
# Visual Comparison Figures
os.makedirs(os.path.join('results', 'plots'), exist_ok=True)

# 1. R² Comparison
plt.figure(figsize=(10, 5))
x_labels = [f"{row['Model']}\n({row['Version']})" for _, row in combined_df.iterrows()]
x_pos = np.arange(len(x_labels))
width = 0.35

plt.bar(x_pos - width/2, combined_df['CV_R2'], width, label='5-Fold CV R²', color='#4A90E2', edgecolor='black')
plt.bar(x_pos + width/2, combined_df['Test_R2'], width, label='Test Set R²', color='#50E3C2', edgecolor='black')
plt.xticks(x_pos, x_labels, fontsize=9)
plt.ylabel('R² Score (Higher is Better)', fontsize=11)
plt.title('Figure 1: Cross-Validation vs Test R² Across Models', fontsize=13, fontweight='bold')
plt.legend(loc='lower right')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig(os.path.join('results', 'plots', 'model_r2_comparison.png'), dpi=300)
plt.show()

# 2. Test Error Comparison
fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()
ax1.bar(x_pos - width/2, combined_df['Test_MAE'], width, label='Log MAE', color='#E67E22', edgecolor='black')
ax2.plot(x_pos + width/2, combined_df['Test_MAE_Dollar'], color='#9B59B6', marker='o', lw=2.5, label='Dollar Scale MAE ($)')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(x_labels, fontsize=9)
ax1.set_ylabel('Test MAE in Log Space', color='#E67E22', fontsize=11)
ax2.set_ylabel('Test MAE in USD ($)', color='#9B59B6', fontsize=11)
plt.title('Figure 2: Test Error Comparison (Log MAE & Dollar MAE)', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.savefig(os.path.join('results', 'plots', 'model_mae_comparison.png'), dpi=300)
plt.show()

---
## 4. Final Champion Model Selection & Diagnostics

In [ ]:
# Identify champion model by test R² and lowest MAE
best_row = combined_df.loc[combined_df['Test_R2'].idxmax()]

print("=" * 65)
print("FINAL CHAMPION MODEL SELECTION")
print("=" * 65)
print(f"Selected Model   : {best_row['Model']} ({best_row['Version']})")
print(f"Parameters       : {best_row['Parameters']}")
print(f"CV R² Performance: {best_row['CV_R2']:.4f}")
print(f"Test R²          : {best_row['Test_R2']:.4f}")
print(f"Test MAE (USD $) : ${best_row['Test_MAE_Dollar']:,.2f}")
print("=" * 65)

# Train champion model and plot actual vs predicted and residuals
from sklearn.ensemble import GradientBoostingRegressor

X_train = pd.read_csv('X_train.csv')
X_test  = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').values.ravel()
y_test  = pd.read_csv('y_test.csv').values.ravel()

champ_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
champ_model.fit(X_train, y_train)
y_pred_champ = champ_model.predict(X_test)
residuals = y_test - y_pred_champ

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred_champ, alpha=0.35, color='#2980B9', edgecolors='none')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Ideal Fit (y=x)')
axes[0].set_title('Figure 3: Actual vs Predicted Log Price', fontweight='bold')
axes[0].set_xlabel('Actual log(1 + price)')
axes[0].set_ylabel('Predicted log(1 + price)')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.5)

# Residual Distribution
axes[1].hist(residuals, bins=35, color='#27AE60', edgecolor='black', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--', lw=2, label='Zero Error')
axes[1].set_title(f'Figure 4: Residuals (Mean: {residuals.mean():.4f}, Std: {residuals.std():.4f})', fontweight='bold')
axes[1].set_xlabel('Residual Error')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join('results', 'plots', 'final_model_actual_vs_predicted.png'), dpi=300)
plt.show()